# Hunyuan3D-2 Full VM/Jupyter Workflow

This notebook is for Google Cloud VM + JupyterLab. It does not use Colab APIs.

- VM workspace: `/home/pminhchien2006/work`
- Jupyter usually uses port 8888. This notebook will not kill ngrok.
- The Hunyuan worker uses port 8010. If Windows backend needs access, open another SSH tab and run `ngrok http 8010`.
- Optional: set `HF_TOKEN` in the VM shell before starting Jupyter if Hugging Face downloads are rate-limited.
- Full model selected: `tencent/Hunyuan3D-2`, subfolder `hunyuan3d-dit-v2-0`.


# Hunyuan3D-2 Full VM Worker: Shape First, Optional Paint

This notebook runs a Hunyuan3D-2 full shape worker on a Google Cloud VM/Tesla T4. Expo can generate shape first, then optionally call Paint texture as a second request.


## 0. Reset VM worker processes

In [ ]:
import subprocess, time
# Stop only the old Hunyuan worker. Do not kill ngrok because it may expose JupyterLab.
for pattern in ['uvicorn hunyuan_vm_worker:app']:
    subprocess.run(f"pkill -f '{pattern}'", shell=True, check=False)
time.sleep(2)
print('Old worker process killed if it existed.')


## 1. Clone Hunyuan3D-2 official repo

In [ ]:
%cd /home/pminhchien2006/work
!rm -rf Hunyuan3D-2
!git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd /home/pminhchien2006/work/Hunyuan3D-2
!git rev-parse --short HEAD

## 2. Install dependencies

Texture modules are still installed, but the default runtime path is shape-only first.

In [ ]:
%cd /home/pminhchien2006/work/Hunyuan3D-2
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -e .

# Required only for the later paint-texture phase.
%cd /home/pminhchien2006/work/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer
!python setup.py install
%cd /home/pminhchien2006/work/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer
!python setup.py install
%cd /home/pminhchien2006/work/Hunyuan3D-2

!python -m pip install fastapi uvicorn python-multipart pyngrok requests trimesh psutil

## 3. GPU, RAM, and import check

In [ ]:
import psutil, torch
ram = psutil.virtual_memory()
print('System RAM GB:', round(ram.used / 1024**3, 2), '/', round(ram.total / 1024**3, 2))
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print('GPU VRAM free/total GB:', round(free / 1024**3, 2), '/', round(total / 1024**3, 2))

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
print('shape import: ok')
try:
    from hy3dgen.texgen import Hunyuan3DPaintPipeline
    print('texture import: ok')
except Exception as exc:
    print('texture import failed:', repr(exc))
    print('Shape-only can still run.')

## 4. Write VM worker

In [ ]:
from pathlib import Path
worker_path = Path('/home/pminhchien2006/work/hunyuan_vm_worker.py')
worker_path.write_text('from __future__ import annotations\n\nimport io\nimport gc\nimport os\nimport ctypes\nimport time\nimport traceback\nfrom pathlib import Path\nfrom threading import Lock, Thread\n\nfrom fastapi import FastAPI, File, Form, HTTPException, UploadFile\nfrom fastapi.responses import Response\nfrom PIL import Image, ImageOps, UnidentifiedImageError\nfrom starlette.concurrency import run_in_threadpool\n\nimport torch\n\n\nWORK_DIR = Path(os.environ.get("HUNYUAN_VM_WORK_DIR", "/tmp/hunyuan_jobs"))\nMODEL_ID = os.environ.get("HUNYUAN_MODEL_ID", "tencent/Hunyuan3D-2mini")\nMODEL_SUBFOLDER = os.environ.get("HUNYUAN_MODEL_SUBFOLDER", "hunyuan3d-dit-v2-mini")\nTEXGEN_MODEL_ID = os.environ.get("HUNYUAN_TEXGEN_MODEL_ID", "tencent/Hunyuan3D-2")\nINFERENCE_STEPS = int(os.environ.get("HUNYUAN_INFERENCE_STEPS", "15"))\nOCTREE_RESOLUTION = int(os.environ.get("HUNYUAN_OCTREE_RESOLUTION", "256"))\nNUM_CHUNKS = int(os.environ.get("HUNYUAN_NUM_CHUNKS", "6000"))\nSEED = int(os.environ.get("HUNYUAN_SEED", "12345"))\nSHAPE_TORCH_DTYPE = os.environ.get("HUNYUAN_SHAPE_TORCH_DTYPE", "float16").strip().lower()\nTEXTURE_TORCH_DTYPE = os.environ.get("HUNYUAN_TEXTURE_TORCH_DTYPE", "float16").strip().lower()\nMIN_TEXTURE_RAM_FREE_GB = float(os.environ.get("HUNYUAN_MIN_TEXTURE_RAM_FREE_GB", "8.5"))\nLOW_CPU_MEM_USAGE = os.environ.get("HUNYUAN_LOW_CPU_MEM_USAGE", "1").strip().lower() in {\n    "1",\n    "true",\n    "yes",\n    "on",\n}\nKEEP_SHAPE_PIPELINE = os.environ.get("HUNYUAN_KEEP_SHAPE_PIPELINE", "0").strip().lower() in {\n    "1",\n    "true",\n    "yes",\n    "on",\n}\nKEEP_TEXTURE_PIPELINE = os.environ.get("HUNYUAN_KEEP_TEXTURE_PIPELINE", "0").strip().lower() in {\n    "1",\n    "true",\n    "yes",\n    "on",\n}\n\nWORK_DIR.mkdir(parents=True, exist_ok=True)\napp = FastAPI(title="Hunyuan VM Worker")\n_shape_pipeline = None\n_texture_pipeline = None\n_busy_lock = Lock()\n_busy_job = None\n_busy_started_at = None\n_jobs_lock = Lock()\n_jobs = {}\n\n\ndef start_busy(job_name: str):\n    global _busy_job\n    global _busy_started_at\n    if not _busy_lock.acquire(blocking=False):\n        raise HTTPException(\n            status_code=409,\n            detail=f"Worker is busy with {_busy_job or \'another job\'}. Wait for it to finish before sending another request.",\n        )\n    _busy_job = job_name\n    _busy_started_at = time.time()\n\n\ndef finish_busy():\n    global _busy_job\n    global _busy_started_at\n    _busy_job = None\n    _busy_started_at = None\n    _busy_lock.release()\n\n\ndef busy_payload():\n    return {\n        "busy": _busy_lock.locked(),\n        "busy_job": _busy_job,\n        "busy_seconds": round(time.time() - _busy_started_at, 1) if _busy_started_at else 0,\n    }\n\n\ndef update_job(job_id: str, **values):\n    with _jobs_lock:\n        job = _jobs.setdefault(job_id, {"job_id": job_id})\n        job.update(values)\n\n\ndef get_job(job_id: str):\n    with _jobs_lock:\n        job = _jobs.get(job_id)\n        return dict(job) if job else None\n\n\ndef public_job(job_id: str):\n    job = get_job(job_id)\n    if not job:\n        raise HTTPException(status_code=404, detail=f"Job not found: {job_id}")\n    return {\n        "job_id": job_id,\n        "kind": job.get("kind"),\n        "status": job.get("status"),\n        "error": job.get("error"),\n        "started_at": job.get("started_at"),\n        "completed_at": job.get("completed_at"),\n        "mesh_ready": bool(job.get("output_path") and Path(job["output_path"]).is_file()),\n        **memory_payload("memory"),\n        **busy_payload(),\n    }\n\n\ndef start_background_job(job_id: str, kind: str, job_name: str, target, *args):\n    start_busy(job_name)\n    update_job(\n        job_id,\n        kind=kind,\n        status="running",\n        error=None,\n        output_path=None,\n        started_at=time.time(),\n        completed_at=None,\n    )\n\n    def runner():\n        try:\n            output_path = target(*args)\n            update_job(\n                job_id,\n                status="done",\n                output_path=str(output_path),\n                completed_at=time.time(),\n            )\n        except BaseException as exc:\n            update_job(\n                job_id,\n                status="error",\n                error=f"{type(exc).__name__}: {exc}",\n                traceback=traceback.format_exc(),\n                completed_at=time.time(),\n            )\n        finally:\n            finish_busy()\n\n    Thread(target=runner, daemon=True).start()\n    return public_job(job_id)\n\n\ndef cleanup_memory():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n        try:\n            torch.cuda.ipc_collect()\n        except Exception:\n            pass\n    try:\n        ctypes.CDLL("libc.so.6").malloc_trim(0)\n    except Exception:\n        pass\n\n\ndef memory_payload(prefix: str = "memory"):\n    payload = {}\n    try:\n        import psutil\n\n        ram = psutil.virtual_memory()\n        payload[f"{prefix}_ram_used_gb"] = round(ram.used / 1024**3, 2)\n        payload[f"{prefix}_ram_total_gb"] = round(ram.total / 1024**3, 2)\n    except Exception:\n        pass\n    if torch.cuda.is_available():\n        free, total = torch.cuda.mem_get_info()\n        payload[f"{prefix}_cuda_free_gb"] = round(free / 1024**3, 2)\n        payload[f"{prefix}_cuda_total_gb"] = round(total / 1024**3, 2)\n        payload[f"{prefix}_cuda_allocated_mb"] = round(torch.cuda.memory_allocated() / 1024 / 1024, 1)\n        payload[f"{prefix}_cuda_reserved_mb"] = round(torch.cuda.memory_reserved() / 1024 / 1024, 1)\n    return payload\n\n\ndef ram_available_gb() -> float | None:\n    try:\n        import psutil\n\n        return psutil.virtual_memory().available / 1024**3\n    except Exception:\n        return None\n\n\ndef require_texture_ram_headroom():\n    cleanup_memory()\n    available = ram_available_gb()\n    if available is None:\n        return\n    if available < MIN_TEXTURE_RAM_FREE_GB:\n        raise RuntimeError(\n            "Not enough VM system RAM to safely load Hunyuan3D Paint. "\n            f"Available={available:.2f}GB, required>={MIN_TEXTURE_RAM_FREE_GB:.2f}GB. "\n            "Restart the VM worker or Jupyter kernel or skip texture for this session."\n        )\n\n\ndef configured_torch_dtype(name: str):\n    if name in {"", "none", "auto"}:\n        return None\n    if name in {"float16", "fp16", "half"}:\n        return torch.float16\n    if name in {"bfloat16", "bf16"}:\n        return torch.bfloat16\n    if name in {"float32", "fp32"}:\n        return torch.float32\n    raise RuntimeError(f"Unsupported torch dtype: {name}")\n\n\ndef move_pipeline_to_device(pipeline, device: str):\n    if device == "cpu" or not hasattr(pipeline, "to"):\n        return pipeline\n    moved = pipeline.to(device)\n    return moved if moved is not None else pipeline\n\n\ndef load_pipeline_with_fallback(\n    pipeline_cls,\n    model_id: str,\n    dtype_name: str,\n    *,\n    subfolder: str | None = None,\n    use_safetensors: bool = True,\n):\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    base_kwargs = {}\n    if use_safetensors:\n        base_kwargs["use_safetensors"] = True\n    if subfolder:\n        base_kwargs["subfolder"] = subfolder\n    minimal_kwargs = {}\n    if subfolder:\n        minimal_kwargs["subfolder"] = subfolder\n\n    optimized_kwargs = dict(base_kwargs)\n    dtype = configured_torch_dtype(dtype_name)\n    if dtype is not None:\n        optimized_kwargs["torch_dtype"] = dtype\n    if LOW_CPU_MEM_USAGE:\n        optimized_kwargs["low_cpu_mem_usage"] = True\n\n    attempts = [\n        (optimized_kwargs, True),\n        ({**optimized_kwargs, "device": device}, False),\n        ({**base_kwargs, "device": device}, False),\n        (base_kwargs, True),\n        ({**minimal_kwargs, "device": device}, False),\n        (minimal_kwargs, True),\n    ]\n    last_exc = None\n    for kwargs, move_after_load in attempts:\n        try:\n            pipeline = pipeline_cls.from_pretrained(model_id, **kwargs)\n            if move_after_load:\n                pipeline = move_pipeline_to_device(pipeline, device)\n            return pipeline\n        except TypeError as exc:\n            last_exc = exc\n            cleanup_memory()\n        except RuntimeError as exc:\n            last_exc = exc\n            cleanup_memory()\n\n    raise RuntimeError(f"Could not load pipeline {model_id}: {last_exc}") from last_exc\n\n\ndef get_shape_pipeline():\n    global _shape_pipeline\n    if _shape_pipeline is None:\n        try:\n            from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline\n        except Exception as exc:\n            raise HTTPException(\n                status_code=503,\n                detail="Hunyuan shape dependencies are not ready. Re-run the install cells and check worker logs.",\n            ) from exc\n\n        print(f"Loading shape model {MODEL_ID}/{MODEL_SUBFOLDER} with low-memory settings...")\n        cleanup_memory()\n        _shape_pipeline = load_pipeline_with_fallback(\n            Hunyuan3DDiTFlowMatchingPipeline,\n            MODEL_ID,\n            subfolder=MODEL_SUBFOLDER,\n            dtype_name=SHAPE_TORCH_DTYPE,\n        )\n    return _shape_pipeline\n\n\ndef get_texture_pipeline():\n    global _texture_pipeline\n    if _texture_pipeline is None:\n        try:\n            from hy3dgen.texgen import Hunyuan3DPaintPipeline\n        except Exception as exc:\n            raise HTTPException(\n                status_code=503,\n                detail=(\n                    "Hunyuan texture dependencies are not ready. Install the official "\n                    "texgen custom_rasterizer and differentiable_renderer modules."\n                ),\n            ) from exc\n\n        print(f"Loading texture model {TEXGEN_MODEL_ID} with low-memory settings...")\n        cleanup_memory()\n        _texture_pipeline = load_pipeline_with_fallback(\n            Hunyuan3DPaintPipeline,\n            TEXGEN_MODEL_ID,\n            dtype_name=TEXTURE_TORCH_DTYPE,\n            use_safetensors=False,\n        )\n    return _texture_pipeline\n\n\ndef unload_shape_pipeline():\n    global _shape_pipeline\n    _shape_pipeline = None\n    cleanup_memory()\n\n\ndef unload_texture_pipeline():\n    global _texture_pipeline\n    _texture_pipeline = None\n    cleanup_memory()\n\n\ndef read_image(image_bytes: bytes) -> Image.Image:\n    try:\n        return ImageOps.exif_transpose(Image.open(io.BytesIO(image_bytes))).convert("RGBA")\n    except UnidentifiedImageError as exc:\n        raise HTTPException(status_code=400, detail="Uploaded file is not a valid image.") from exc\n\n\n@app.get("/health")\ndef health():\n    return {\n        "status": "ok",\n        "cuda_available": torch.cuda.is_available(),\n        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",\n        "model_id": MODEL_ID,\n        "model_subfolder": MODEL_SUBFOLDER,\n        "texgen_model_id": TEXGEN_MODEL_ID,\n        "shape_pipeline_loaded": _shape_pipeline is not None,\n        "texture_pipeline_loaded": _texture_pipeline is not None,\n        "min_texture_ram_free_gb": MIN_TEXTURE_RAM_FREE_GB,\n        "cuda_memory_allocated_mb": (\n            round(torch.cuda.memory_allocated() / 1024 / 1024, 1) if torch.cuda.is_available() else None\n        ),\n        "cuda_memory_reserved_mb": (\n            round(torch.cuda.memory_reserved() / 1024 / 1024, 1) if torch.cuda.is_available() else None\n        ),\n        **busy_payload(),\n    }\n\n\n@app.post("/cleanup-memory")\ndef cleanup_memory_endpoint(\n    unload_shape: bool = True,\n    unload_texture: bool = True,\n    clear_jobs: bool = False,\n):\n    global _shape_pipeline\n    global _texture_pipeline\n    if _busy_lock.locked():\n        raise HTTPException(\n            status_code=409,\n            detail=f"Worker is busy with {_busy_job or \'another job\'}. Wait before cleanup.",\n        )\n\n    before = memory_payload("before")\n    if unload_shape:\n        _shape_pipeline = None\n    if unload_texture:\n        _texture_pipeline = None\n    if clear_jobs:\n        with _jobs_lock:\n            _jobs.clear()\n    cleanup_memory()\n    after = memory_payload("after")\n    return {\n        "status": "ok",\n        "shape_pipeline_loaded": _shape_pipeline is not None,\n        "texture_pipeline_loaded": _texture_pipeline is not None,\n        **before,\n        **after,\n        **busy_payload(),\n    }\n\n\n@app.get("/jobs/{job_id}")\ndef job_status(job_id: str):\n    return public_job(job_id)\n\n\n@app.get("/jobs/{job_id}/mesh")\ndef job_mesh(job_id: str):\n    job = get_job(job_id)\n    if not job:\n        raise HTTPException(status_code=404, detail=f"Job not found: {job_id}")\n    if job.get("status") != "done":\n        raise HTTPException(status_code=409, detail=f"Job is not done: {job.get(\'status\')}")\n    output_path = Path(job.get("output_path") or "")\n    if not output_path.is_file():\n        raise HTTPException(status_code=404, detail=f"Mesh output not found for job: {job_id}")\n    return glb_response(output_path)\n\n\n@app.post("/warmup")\ndef warmup():\n    start_busy("warmup-shape")\n    try:\n        get_shape_pipeline()\n        return health()\n    finally:\n        finish_busy()\n\n\n@app.post("/warmup-texture")\ndef warmup_texture():\n    start_busy("warmup-texture")\n    try:\n        require_texture_ram_headroom()\n        get_texture_pipeline()\n        return health()\n    finally:\n        finish_busy()\n\n\nasync def prepare_job_image(image: UploadFile, job_id: str) -> tuple[Image.Image, Path]:\n    image_bytes = await image.read()\n    if not image_bytes:\n        raise HTTPException(status_code=400, detail="Uploaded image is empty.")\n\n    pil_image = read_image(image_bytes)\n    job_dir = WORK_DIR / job_id\n    job_dir.mkdir(parents=True, exist_ok=True)\n    input_path = job_dir / "input.png"\n    pil_image.save(input_path)\n    return pil_image, job_dir\n\n\ndef generate_shape_mesh(pil_image: Image.Image):\n    pipeline = get_shape_pipeline()\n    with torch.inference_mode():\n        return pipeline(\n            image=pil_image,\n            num_inference_steps=INFERENCE_STEPS,\n            octree_resolution=OCTREE_RESOLUTION,\n            num_chunks=NUM_CHUNKS,\n            generator=torch.manual_seed(SEED),\n        )[0]\n\n\ndef validate_glb_output(output_format: str):\n    if output_format.lower() != "glb":\n        raise HTTPException(status_code=400, detail="Only glb output is supported.")\n\n\ndef load_mesh(mesh_path: Path):\n    try:\n        import trimesh\n    except Exception as exc:\n        raise HTTPException(status_code=503, detail=f"trimesh is not importable: {exc}") from exc\n\n    mesh = trimesh.load(mesh_path, force="mesh")\n    if hasattr(mesh, "geometry"):\n        mesh = mesh.dump(concatenate=True)\n    return mesh\n\n\ndef glb_response(output_path: Path, filename: str = "mesh.glb") -> Response:\n    content = output_path.read_bytes()\n    return Response(\n        content=content,\n        media_type="model/gltf-binary",\n        headers={\n            "Content-Disposition": f\'attachment; filename="{filename}"\',\n            "Content-Length": str(len(content)),\n        },\n    )\n\n\ndef run_shape_job(pil_image: Image.Image, job_dir: Path) -> Path:\n    mesh = None\n    try:\n        mesh = generate_shape_mesh(pil_image)\n        output_path = job_dir / "mesh.glb"\n        mesh.export(output_path)\n        return output_path\n    finally:\n        if mesh is not None:\n            del mesh\n        if not KEEP_SHAPE_PIPELINE:\n            unload_shape_pipeline()\n        else:\n            cleanup_memory()\n\n\ndef run_texture_job(pil_image: Image.Image, mesh_path: Path, job_dir: Path) -> Path:\n    input_mesh = None\n    textured_mesh = None\n    try:\n        require_texture_ram_headroom()\n        input_mesh = load_mesh(mesh_path)\n        texture_pipeline = get_texture_pipeline()\n        with torch.inference_mode():\n            textured_mesh = texture_pipeline(input_mesh, image=pil_image.convert("RGB"))\n    except RuntimeError as exc:\n        cleanup_memory()\n        raise HTTPException(status_code=500, detail=f"Hunyuan texture generation failed: {exc}") from exc\n\n    try:\n        output_path = job_dir / "mesh.glb"\n        textured_mesh.export(output_path)\n        return output_path\n    finally:\n        if input_mesh is not None:\n            del input_mesh\n        if textured_mesh is not None:\n            del textured_mesh\n        if not KEEP_TEXTURE_PIPELINE:\n            unload_texture_pipeline()\n        else:\n            cleanup_memory()\n\n\ndef run_textured_shape_job(pil_image: Image.Image, job_dir: Path) -> Path:\n    mesh = None\n    textured_mesh = None\n    try:\n        mesh = generate_shape_mesh(pil_image)\n        shape_path = job_dir / "shape_mesh.glb"\n        mesh.export(shape_path)\n        if not KEEP_SHAPE_PIPELINE:\n            unload_shape_pipeline()\n        else:\n            cleanup_memory()\n\n        require_texture_ram_headroom()\n        texture_pipeline = get_texture_pipeline()\n        with torch.inference_mode():\n            textured_mesh = texture_pipeline(mesh, image=pil_image.convert("RGB"))\n    except RuntimeError as exc:\n        cleanup_memory()\n        raise HTTPException(status_code=500, detail=f"Hunyuan texture generation failed: {exc}") from exc\n\n    try:\n        output_path = job_dir / "mesh.glb"\n        textured_mesh.export(output_path)\n        return output_path\n    finally:\n        if mesh is not None:\n            del mesh\n        if textured_mesh is not None:\n            del textured_mesh\n        if not KEEP_TEXTURE_PIPELINE:\n            unload_texture_pipeline()\n        else:\n            cleanup_memory()\n\n\n@app.post("/generate-shape")\nasync def generate_shape(\n    image: UploadFile = File(...),\n    job_id: str = Form(...),\n    output_format: str = Form(default="glb"),\n):\n    validate_glb_output(output_format)\n    pil_image, job_dir = await prepare_job_image(image, job_id)\n\n    start_busy(f"shape:{job_id}")\n    try:\n        output_path = await run_in_threadpool(run_shape_job, pil_image, job_dir)\n    finally:\n        finish_busy()\n    return glb_response(output_path)\n\n\n@app.post("/start-shape")\nasync def start_shape(\n    image: UploadFile = File(...),\n    job_id: str = Form(...),\n    output_format: str = Form(default="glb"),\n):\n    validate_glb_output(output_format)\n    pil_image, job_dir = await prepare_job_image(image, job_id)\n    return start_background_job(job_id, "shape", f"shape:{job_id}", run_shape_job, pil_image, job_dir)\n\n\n@app.post("/generate-texture")\nasync def generate_texture(\n    image: UploadFile = File(...),\n    mesh: UploadFile = File(...),\n    job_id: str = Form(...),\n    output_format: str = Form(default="glb"),\n):\n    validate_glb_output(output_format)\n    pil_image, job_dir = await prepare_job_image(image, job_id)\n\n    mesh_bytes = await mesh.read()\n    if not mesh_bytes:\n        raise HTTPException(status_code=400, detail="Uploaded mesh is empty.")\n\n    mesh_path = job_dir / "input_mesh.glb"\n    mesh_path.write_bytes(mesh_bytes)\n\n    start_busy(f"texture:{job_id}")\n    try:\n        output_path = await run_in_threadpool(run_texture_job, pil_image, mesh_path, job_dir)\n    finally:\n        finish_busy()\n    return glb_response(output_path)\n\n\n@app.post("/start-texture")\nasync def start_texture(\n    image: UploadFile = File(...),\n    mesh: UploadFile = File(...),\n    job_id: str = Form(...),\n    output_format: str = Form(default="glb"),\n):\n    validate_glb_output(output_format)\n    pil_image, job_dir = await prepare_job_image(image, job_id)\n\n    mesh_bytes = await mesh.read()\n    if not mesh_bytes:\n        raise HTTPException(status_code=400, detail="Uploaded mesh is empty.")\n\n    mesh_path = job_dir / "input_mesh.glb"\n    mesh_path.write_bytes(mesh_bytes)\n    return start_background_job(job_id, "texture", f"texture:{job_id}", run_texture_job, pil_image, mesh_path, job_dir)\n\n\n@app.post("/generate-textured-shape")\nasync def generate_textured_shape(\n    image: UploadFile = File(...),\n    job_id: str = Form(...),\n    output_format: str = Form(default="glb"),\n):\n    validate_glb_output(output_format)\n    pil_image, job_dir = await prepare_job_image(image, job_id)\n\n    start_busy(f"textured-shape:{job_id}")\n    try:\n        output_path = await run_in_threadpool(run_textured_shape_job, pil_image, job_dir)\n    finally:\n        finish_busy()\n    return glb_response(output_path)\n\n\n@app.post("/start-textured-shape")\nasync def start_textured_shape(\n    image: UploadFile = File(...),\n    job_id: str = Form(...),\n    output_format: str = Form(default="glb"),\n):\n    validate_glb_output(output_format)\n    pil_image, job_dir = await prepare_job_image(image, job_id)\n    return start_background_job(\n        job_id,\n        "textured-shape",\n        f"textured-shape:{job_id}",\n        run_textured_shape_job,\n        pil_image,\n        job_dir,\n    )\n', encoding='utf-8')
print('wrote', worker_path)


## 5. Start worker on port 8010 with low-memory shape settings

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

# Full Hunyuan3D-2 shape model. On Tesla T4 16GB this is close to the limit, so keep settings conservative.
os.environ['HUNYUAN_MODEL_ID'] = 'tencent/Hunyuan3D-2'
os.environ['HUNYUAN_MODEL_SUBFOLDER'] = 'hunyuan3d-dit-v2-0'
os.environ['HUNYUAN_TEXGEN_MODEL_ID'] = 'tencent/Hunyuan3D-2'
os.environ['HUNYUAN_SHAPE_TORCH_DTYPE'] = 'float16'
os.environ['HUNYUAN_TEXTURE_TORCH_DTYPE'] = 'float16'
os.environ['HUNYUAN_LOW_CPU_MEM_USAGE'] = '1'
os.environ['HUNYUAN_MIN_TEXTURE_RAM_FREE_GB'] = '8.5'
os.environ['HUNYUAN_KEEP_SHAPE_PIPELINE'] = '0'
os.environ['HUNYUAN_KEEP_TEXTURE_PIPELINE'] = '0'
os.environ['HUNYUAN_INFERENCE_STEPS'] = '25'
os.environ['HUNYUAN_OCTREE_RESOLUTION'] = '256'
os.environ['HUNYUAN_NUM_CHUNKS'] = '6000'

hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN detected from environment.')
else:
    print('HF_TOKEN not found in environment. Downloads may be slow or rate-limited.')

if 'worker_proc' in globals() and worker_proc.poll() is None:
    worker_proc.terminate()
    time.sleep(2)
subprocess.run("pkill -f 'uvicorn hunyuan_vm_worker:app'", shell=True, check=False)
time.sleep(2)

log_path = Path('/tmp/hunyuan_worker.log')
log_file = log_path.open('w')
worker_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'hunyuan_vm_worker:app', '--host', '0.0.0.0', '--port', '8010'],
    cwd='/home/pminhchien2006/work',
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
print('worker pid:', worker_proc.pid)
print('shape model:', os.environ['HUNYUAN_MODEL_ID'], os.environ['HUNYUAN_MODEL_SUBFOLDER'])
print('texture model:', os.environ['HUNYUAN_TEXGEN_MODEL_ID'])

import requests
for _ in range(60):
    try:
        r = requests.get('http://127.0.0.1:8010/health', timeout=2)
        if r.status_code == 200:
            print('health:', r.text)
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print('Worker did not become healthy. Last log lines:')
    print(log_path.read_text(encoding='utf-8', errors='replace')[-4000:])


## 6. Verify local worker health

In [ ]:
import requests
r = requests.get('http://127.0.0.1:8010/health', timeout=20)
print(r.status_code)
print(r.text)
r.raise_for_status()

## 7. Open ngrok tunnel to VM worker

In [ ]:
import os, requests

# To expose the worker, open another SSH-in-browser tab and run:
#   ngrok http 8010
# Then paste the forwarding URL here, for example: https://xxxx.ngrok-free.dev
PUBLIC_URL = os.environ.get('HUNYUAN_PUBLIC_URL', '').strip().rstrip('/')
if not PUBLIC_URL:
    PUBLIC_URL = input('Paste worker ngrok URL for port 8010: ').strip().rstrip('/')

if not PUBLIC_URL.startswith('https://'):
    raise RuntimeError('Expected an https ngrok URL')

r = requests.get(PUBLIC_URL + '/health', headers={'ngrok-skip-browser-warning': 'true'}, timeout=60)
print('PUBLIC_URL=', PUBLIC_URL)
print('remote health:', r.status_code, r.text)
r.raise_for_status()


## 8. Print Windows backend commands: shape-only default

Copy these commands to Windows PowerShell. Restart the backend every time `PUBLIC_URL` changes.

In [ ]:
try:
    url = PUBLIC_URL
except NameError:
    url = 'https://PASTE_PUBLIC_URL_HERE'

print('Run this in Windows PowerShell for PHASE 1: shape-only')
print()
print(r'cd C:\Users\pminh\Desktop\MyProject\AI_3D_Reconstruction_Systerm_TangDien02')
print(r'.\.venv\Scripts\Activate.ps1')
print('$env:RECONSTRUCTION_BACKEND="hunyuan_remote"')
print(f'$env:HUNYUAN_REMOTE_URL="{url}"')
print('$env:HUNYUAN_REMOTE_OUTPUT_FORMAT="glb"')
print('$env:HUNYUAN_REMOTE_ENABLE_TEXTURE="false"')
print('python -m uvicorn server.main:app --host 0.0.0.0 --port 8000')
print()
print('After Expo creates a shape, copy its job_id and run PHASE 2 texture paint:')
print('curl.exe -X POST "http://127.0.0.1:8000/paint-texture" -F "job_id=<JOB_ID_FROM_EXPO_OR_RESPONSE>"')

## 9. Print Expo local commands

In [ ]:
BACKEND_LAN_IP = '192.168.1.6'  # Change this if your Windows LAN IP changes.
BACKEND_PORT = 8000
api_url = f'http://{BACKEND_LAN_IP}:{BACKEND_PORT}'

print('Run this in a second Windows PowerShell window:')
print()
print(r'cd C:\Users\pminh\Desktop\MyProject\AI_3D_Reconstruction_Systerm_TangDien02\mobile')
print(f'$env:EXPO_PUBLIC_API_BASE_URL="{api_url}"')
print('npm install')
print('npm start')
print()
print('iPhone Safari test before Expo:')
print(f'{api_url}/health')

## 10. Direct worker tests through ngrok

In [ ]:
import requests
try:
    url = PUBLIC_URL
except NameError:
    url = input('Paste PUBLIC_URL: ').strip().rstrip('/')
r = requests.get(url + '/health', headers={'ngrok-skip-browser-warning': 'true'}, timeout=60)
print(r.status_code)
print(r.text)
r.raise_for_status()

In [ ]:
# Shape-only direct test through ngrok using the async job API.
# This does not involve Windows backend or Expo.
import requests, time
from pathlib import Path
try:
    url = PUBLIC_URL
except NameError:
    url = input('Paste PUBLIC_URL: ').strip().rstrip('/')

candidates = list(Path('/home/pminhchien2006/work/Hunyuan3D-2').rglob('*.png')) + list(Path('/home/pminhchien2006/work/Hunyuan3D-2').rglob('*.jpg')) + list(Path('/home/pminhchien2006/work/Hunyuan3D-2').rglob('*.jpeg'))
if not candidates:
    raise FileNotFoundError('No test image found. Upload an image and set TEST_IMAGE manually.')
TEST_IMAGE = str(candidates[0])
remote_job_id = 'direct_shape_test'
print('Using image:', TEST_IMAGE)

with open(TEST_IMAGE, 'rb') as f:
    r = requests.post(
        url + '/start-shape',
        headers={'ngrok-skip-browser-warning': 'true'},
        files={'image': (Path(TEST_IMAGE).name, f, 'image/png')},
        data={'job_id': remote_job_id, 'output_format': 'glb'},
        timeout=120,
    )
print('start:', r.status_code, r.text[:1000])
r.raise_for_status()

while True:
    status = requests.get(
        url + f'/jobs/{remote_job_id}',
        headers={'ngrok-skip-browser-warning': 'true'},
        timeout=60,
    )
    print('status:', status.status_code, status.text[:1000])
    status.raise_for_status()
    payload = status.json()
    if payload.get('status') == 'done':
        break
    if payload.get('status') == 'error':
        raise RuntimeError(payload.get('error'))
    time.sleep(5)

mesh = requests.get(
    url + f'/jobs/{remote_job_id}/mesh',
    headers={'ngrok-skip-browser-warning': 'true'},
    timeout=300,
)
print('mesh:', mesh.status_code, mesh.headers.get('content-type'), mesh.headers.get('content-length'))
mesh.raise_for_status()
out = Path('/home/pminhchien2006/work/direct_shape_test.glb')
out.write_bytes(mesh.content)
print('saved:', out, 'bytes:', out.stat().st_size)

## 11. Cleanup RAM before texture warmup

Run this after shape-only works and before loading the texture pipeline. It unloads the shape pipeline inside the worker process and asks Linux/Python to return free memory.


In [ ]:
import requests
r = requests.post('http://127.0.0.1:8010/cleanup-memory', params={'unload_shape': 'true', 'unload_texture': 'true'}, timeout=60)
print(r.status_code)
print(r.text)


## 12. Optional texture warmup before tapping Paint texture in Expo

Run this once after shape-only is proven to work and RAM has been cleaned. If the worker reports not enough system RAM, skip texture for this session instead of forcing a crash.


In [ ]:
import psutil, requests
ram = psutil.virtual_memory()
print('System RAM GB before texture warmup:', round(ram.used / 1024**3, 2), '/', round(ram.total / 1024**3, 2))

r = requests.post('http://127.0.0.1:8010/warmup-texture', timeout=1800)
print(r.status_code)
print(r.text)
r.raise_for_status()

## 13. Optional direct texture test after warmup-texture succeeds

Run this only after the warmup-texture cell is done and local `/health` is still responsive.

In [ ]:
import requests, time
from pathlib import Path

mesh_path = Path('/home/pminhchien2006/work/direct_shape_test.glb')
if not mesh_path.exists():
    raise FileNotFoundError('Run the direct shape-only test first to create /home/pminhchien2006/work/direct_shape_test.glb')

img_candidates = list(Path('/home/pminhchien2006/work/Hunyuan3D-2').rglob('*.png')) + list(Path('/home/pminhchien2006/work/Hunyuan3D-2').rglob('*.jpg')) + list(Path('/home/pminhchien2006/work/Hunyuan3D-2').rglob('*.jpeg'))
if not img_candidates:
    raise FileNotFoundError('No test image found under /home/pminhchien2006/work/Hunyuan3D-2')
img_path = img_candidates[0]
remote_job_id = 'direct_texture_probe'

with open(img_path, 'rb') as image_file, open(mesh_path, 'rb') as mesh_file:
    r = requests.post(
        'http://127.0.0.1:8010/start-texture',
        files={
            'image': (img_path.name, image_file, 'image/png'),
            'mesh': ('direct_shape_test.glb', mesh_file, 'model/gltf-binary'),
        },
        data={'job_id': remote_job_id, 'output_format': 'glb'},
        timeout=120,
    )
print('start:', r.status_code, r.text[:1000])
r.raise_for_status()

while True:
    status = requests.get(f'http://127.0.0.1:8010/jobs/{remote_job_id}', timeout=60)
    print('status:', status.status_code, status.text[:1000])
    status.raise_for_status()
    payload = status.json()
    if payload.get('status') == 'done':
        break
    if payload.get('status') == 'error':
        raise RuntimeError(payload.get('error'))
    time.sleep(5)

mesh = requests.get(f'http://127.0.0.1:8010/jobs/{remote_job_id}/mesh', timeout=300)
print('mesh:', mesh.status_code, mesh.headers.get('content-type'), mesh.headers.get('content-length'))
mesh.raise_for_status()
out = Path('/home/pminhchien2006/work/direct_texture_probe.glb')
out.write_bytes(mesh.content)
print('saved:', out, 'bytes:', out.stat().st_size)

## 14. Debug RAM and worker logs


In [ ]:
import psutil, torch
from pathlib import Path
ram = psutil.virtual_memory()
print('System RAM GB:', round(ram.used / 1024**3, 2), '/', round(ram.total / 1024**3, 2))
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print('GPU VRAM free/total GB:', round(free / 1024**3, 2), '/', round(total / 1024**3, 2))
log_path = Path('/tmp/hunyuan_worker.log')
print(log_path.read_text(encoding='utf-8', errors='replace')[-12000:] if log_path.exists() else 'No worker log found')